## Check the setup and connect to the database

In [ ]:
%run "../../scripts/01-check_setup.ipynb"

## Use HANA DataFrame

A table with data already exist in your SAP HANA database, so you use [the `table()` method](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/latest/en-US/hana_ml.dataframe.html#hana_ml.dataframe.ConnectionContext.table) to instantiate a HANA DataFrame from the existing database table. 

In [ ]:
hdf_events=myconn.table('CODECONNECT', schema='VECTORS')

## Generating Text Embeddings in SAP HANA Cloud

In [ ]:
content_column = 'content_html'

In [ ]:
print(f"""Number of documents selected for further processing: {hdf_events.count()}""")

In [ ]:
### Generating Text Embeddings in SAP HANA Cloud with the new PAL function, function available with hana-ml 2.23.
from hana_ml.text.pal_embeddings import PALEmbeddings
pe = PALEmbeddings(model_version="SAP_GXY.20250407")

hdf_events_docs = pe.fit_transform(hdf_events.add_id(), key="ID", target=[f"{content_column}"], thread_number=10, batch_size=10) #, max_token_num=512
print(f"{hdf_events_docs.count()} records processed in {round(pe.runtime, 3)} sec")

In [ ]:
hdf_events_docs.get_table_structure()

In [ ]:
hdf_events_docs.head(1).collect()

In [ ]:
print(hdf_events_docs.select_statement)

In [ ]:
hdf_events_docs=hdf_events_docs.save(where="#CODECONNECT_DOCS_EMBEDDINGS", force=True)

In [ ]:
print(hdf_events_docs.select_statement)

## Semantic search in FAQ

In [ ]:
prompt='Who will present a session about integration of Predictive Analytic Library with AI Agents? '

In [ ]:
df_result = myconn.sql(
    f"""SELECT
    COSINE_SIMILARITY(VECTOR_EMBEDDING('{prompt}', 'QUERY', 'SAP_GXY.20250407'), "VECTOR_COL_{content_column}") AS "SIMILARITY",
    "ID", "{content_column}", "metadata"
    FROM ({hdf_events_docs.select_statement})
    ORDER BY 1 DESC;
    """
).collect()

In [ ]:
df_result.head(3)

In [ ]:
# Print the rows of the 'content' column
print(df_result['metadata'][0])